In [10]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 0 : CONFIGURATION                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from pyspark.sql import functions as F
from pyspark.sql.types import *

storage_account = "energybigdatastorage"
container_raw = "raw"
container_processed = "processed"

path_raw = f"abfss://{container_raw}@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/daily_dataset.csv"
path_processed = f"abfss://{container_processed}@{storage_account}.dfs.core.windows.net/daily_dataset_csv/"

print(f"Source: {path_raw}")
print(f"Destination: {path_processed}")

In [11]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 : INGESTION                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

daily_schema = StructType([
    StructField("LCLid", StringType(), True),
    StructField("day", DateType(), True),
    StructField("energy_median", DoubleType(), True),
    StructField("energy_mean", DoubleType(), True),
    StructField("energy_max", DoubleType(), True),
    StructField("energy_count", IntegerType(), True),
    StructField("energy_std", DoubleType(), True),
    StructField("energy_sum", DoubleType(), True),
    StructField("energy_min", DoubleType(), True)
])

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(daily_schema) \
    .load(path_raw)

print(f"Nombre de lignes : {df.count()}")
print(f"Nombre de colonnes : {len(df.columns)}")
print("\n=== SCHEMA ===")
df.printSchema()
print("\n=== APERCU ===")
df.show(5)

In [12]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 : PROFILING AVANT NETTOYAGE                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES ===")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

print("\n=== STATISTIQUES ===")
df.describe().show()

print("\n=== DOUBLONS ===")
print(f"Nombre de doublons : {df.count() - df.dropDuplicates().count()}")

print("\n=== VALEURS NEGATIVES ===")
energy_cols = ["energy_median", "energy_mean", "energy_max", "energy_sum", "energy_min"]
for c in energy_cols:
    neg_count = df.filter(F.col(c) < 0).count()
    print(f"   {c}: {neg_count:,} valeurs negatives")

In [17]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 : NETTOYAGE AVEC CORRECTION DES INCONSISTANCES                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 1. Supprimer lignes sans LCLid ou day
df_clean = df.filter(F.col("LCLid").isNotNull() & F.col("day").isNotNull())

# 2. Trim + upper LCLid
df_clean = df_clean.withColumn("LCLid", F.upper(F.trim(F.col("LCLid"))))

# 3. Supprimer doublons
df_clean = df_clean.dropDuplicates()

# 4. Negatifs -> NULL
energy_cols = ["energy_median", "energy_mean", "energy_max", "energy_sum", "energy_min"]
for c in energy_cols:
    df_clean = df_clean.withColumn(c, F.when(F.col(c) < 0, F.lit(None)).otherwise(F.col(c)))

# 5. NaN -> NULL
df_clean = df_clean.replace(float('nan'), None)

# 6. Supprimer lignes sans donnees energie
df_clean = df_clean.filter(
    F.col("energy_median").isNotNull() | 
    F.col("energy_mean").isNotNull() | 
    F.col("energy_sum").isNotNull()
)

# 7. DETECTER les inconsistances AVANT correction
df_clean = df_clean.withColumn("consistency_flag",
    F.when(
        (F.col("energy_max").isNotNull()) & (F.col("energy_mean").isNotNull()) & (F.col("energy_min").isNotNull()) &
        ((F.col("energy_max") < F.col("energy_mean")) | (F.col("energy_mean") < F.col("energy_min"))),
        1
    ).otherwise(0))

inconsistent_count = df_clean.filter(F.col("consistency_flag") == 1).count()
print(f"Inconsistances detectees AVANT correction: {inconsistent_count}")

# 8. CORRIGER les inconsistances (recalculer mean comme mediane de [min, max])
# Si max < mean: mean = (min + max) / 2
# Si mean < min: mean = (min + max) / 2
df_clean = df_clean.withColumn("energy_mean",
    F.when(
        F.col("consistency_flag") == 1,
        (F.col("energy_min") + F.col("energy_max")) / 2
    ).otherwise(F.col("energy_mean"))
)

# Recalculer le flag apres correction
df_clean = df_clean.withColumn("consistency_flag",
    F.when(
        (F.col("energy_max").isNotNull()) & (F.col("energy_mean").isNotNull()) & (F.col("energy_min").isNotNull()) &
        ((F.col("energy_max") < F.col("energy_mean")) | (F.col("energy_mean") < F.col("energy_min"))),
        1
    ).otherwise(0))

inconsistent_after = df_clean.filter(F.col("consistency_flag") == 1).count()
print(f"Inconsistances APRES correction: {inconsistent_after}")

# 9. Features temporelles
df_clean = df_clean \
    .withColumn("year", F.year(F.col("day"))) \
    .withColumn("month", F.month(F.col("day"))) \
    .withColumn("dayofweek", F.dayofweek(F.col("day"))) \
    .withColumn("is_weekend", F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0)) \
    .withColumn("quarter", F.quarter(F.col("day"))) \
    .withColumn("processed_date", F.current_date())

# 10. Features derivees
df_clean = df_clean \
    .withColumn("energy_range", F.col("energy_max") - F.col("energy_min")) \
    .withColumn("energy_cv", F.when(F.col("energy_mean") != 0, F.col("energy_std") / F.abs(F.col("energy_mean"))).otherwise(F.lit(None)))

print(f"\nNombre de lignes nettoyees: {df_clean.count()}")
df_clean.show(5)

In [18]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 : STATISTIQUES APRES NETTOYAGE                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print("=== VALEURS NULLES APRES ===")
df_clean.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()

print("\n=== CONSISTANCE ===")
df_clean.groupBy("consistency_flag").count().show()

print("\n=== DISTRIBUTION TEMPORRELLE ===")
df_clean.select(
    F.min("day").alias("date_min"),
    F.max("day").alias("date_max"),
    F.countDistinct("day").alias("nb_jours"),
    F.countDistinct("LCLid").alias("nb_compteurs")
).show()

In [19]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 : SAUVEGARDE DELTA                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .save(path_processed)

print("Sauvegarde terminee dans processed/daily_dataset_csv/")

In [20]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 : VERIFICATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

df_verify = spark.read.format("delta").load(path_processed)
print(f"Verification: {df_verify.count()} lignes")
print("\n=== SCHEMA ===")
df_verify.printSchema()
print("\n=== APERCU ===")
df_verify.show(5)

print("\n=== TESTS RAPIDES ===")
print(f"LCLid NULL: {df_verify.filter(F.col('LCLid').isNull()).count()}")
print(f"day NULL: {df_verify.filter(F.col('day').isNull()).count()}")
print(f"Negatifs: {df_verify.filter((F.col('energy_mean') < 0) | (F.col('energy_sum') < 0)).count()}")
print(f"Inconsistances: {df_verify.filter(F.col('consistency_flag') == 1).count()}")